# Notebook 04 — Pricing & Discount Analysis

**Amazon Market Intelligence**  
**Questions answered:** Q4 (What's the price sweet spot?), Q5 (Who are my competitors and how do they win?)  
**Tool mode:** Competitive Positioning  
**Gold tables:** `gold_price_positioning` (~1,200 rows), `gold_discount_effectiveness` (807 rows)

---

## 0 — Setup

In [1]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os

DB_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'amazon_intelligence.duckdb')
con = duckdb.connect(DB_PATH, read_only=True)

CHARTS_DIR = 'charts/04_pricing_discounts'
os.makedirs(CHARTS_DIR, exist_ok=True)

TEMPLATE = 'plotly_white'

def save_chart(fig, name):
    fig.write_html(f'{CHARTS_DIR}/{name}.html')
    try:
        fig.write_image(f'{CHARTS_DIR}/{name}.png', width=1200, height=700, scale=2)
    except Exception:
        pass

print(f'Connected to: {DB_PATH}')
print(f'Charts → {CHARTS_DIR}/')

Connected to: c:\Users\thinkpad\Desktop\amazon-market-intelligence\data\amazon_intelligence.duckdb
Charts → charts/04_pricing_discounts/


## 1 — Schema Discovery

In [2]:
for table in ['gold_price_positioning', 'gold_discount_effectiveness']:
    print(f'\n=== {table} ===')
    print(con.sql(f'DESCRIBE {table}').df().to_string())
    print(f'Rows: {con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0]:,}')


=== gold_price_positioning ===
                column_name column_type null   key default extra
0               subcategory     VARCHAR  YES  None    None  None
1                price_tier     VARCHAR  YES  None    None  None
2             product_count      BIGINT  YES  None    None  None
3                 avg_price      DOUBLE  YES  None    None  None
4              median_price      DOUBLE  YES  None    None  None
5                avg_rating      DOUBLE  YES  None    None  None
6               avg_reviews      DOUBLE  YES  None    None  None
7          total_units_sold     HUGEINT  YES  None    None  None
8             total_revenue      DOUBLE  YES  None    None  None
9   avg_revenue_per_product      DOUBLE  YES  None    None  None
10               pct_active      DOUBLE  YES  None    None  None
11         pct_best_sellers      DOUBLE  YES  None    None  None
12         avg_discount_pct      DOUBLE  YES  None    None  None
13              pct_branded      DOUBLE  YES  None    None

## 2 — Load Gold Tables

In [3]:
df_price = con.sql('SELECT * FROM gold_price_positioning').df()
df_disc = con.sql('SELECT * FROM gold_discount_effectiveness').df()

print(f'Price positioning: {df_price.shape}')
print(f'Discount effectiveness: {df_disc.shape}')

df_price.head(3)

Price positioning: (1229, 16)
Discount effectiveness: (807, 12)


,subcategory,price_tier,product_count,avg_price,median_price,avg_rating,avg_reviews,total_units_sold,total_revenue,avg_revenue_per_product,pct_active,pct_best_sellers,avg_discount_pct,pct_branded,avg_title_length,pct_with_features
0,Abrasive & Finishing Products,Low,4010,16.20,15.58,3.57,0.7,84750.0,1379178.0,343.93,17.6,0.2,16.0,25.0,126.5,29.7
1,Abrasive & Finishing Products,Mid,1346,34.98,33.95,3.48,0.0,15150.0,499403.0,371.03,11.3,0.1,15.5,21.4,113.3,26.5
2,Abrasive & Finishing Products,Budget,2555,7.55,7.99,3.38,0.0,58050.0,437358.0,171.18,15.7,0.4,20.9,22.0,118.2,27.3


In [4]:
df_disc.head(3)

,subcategory,discount_tier,product_count,avg_discount,avg_price,avg_rating,avg_reviews,total_units_sold,total_revenue,avg_revenue_per_product,pct_active,pct_best_sellers
0,Abrasive & Finishing Products,Light (1-19%),802,10.6,24.26,3.94,0.0,16600.0,314932.5,392.68,16.5,0.4
1,Abrasive & Finishing Products,Medium (20-49%),317,30.2,19.82,4.09,0.0,13150.0,152438.0,480.88,24.6,0.3
2,Abrasive & Finishing Products,Deep (50%+),43,56.2,11.24,3.50,0.0,1100.0,9719.0,226.02,14.0,0.0


---

## 3 — Price Tier Landscape

How does revenue distribute across price tiers? The conventional wisdom says "cheap wins."  
Finding #21 already proved this false in Kitchen & Dining. Let's see if it holds across all categories.

In [5]:
SUBCAT_COL = 'subcategory'          # FIX ME
TIER_COL = 'price_tier'             # FIX ME
TIER_REVENUE = 'total_revenue'      # FIX ME
TIER_PRODUCTS = 'product_count'     # FIX ME
TIER_AVG_SALES = 'avg_units_sold'   # FIX ME
TIER_AVG_RATING = 'avg_rating'      # FIX ME

print(df_price.columns.tolist())

['subcategory', 'price_tier', 'product_count', 'avg_price', 'median_price', 'avg_rating', 'avg_reviews', 'total_units_sold', 'total_revenue', 'avg_revenue_per_product', 'pct_active', 'pct_best_sellers', 'avg_discount_pct', 'pct_branded', 'avg_title_length', 'pct_with_features']


### 3.1 — Overall Revenue by Price Tier

Aggregate across all categories: which price tier captures the most money?

In [6]:
tier_totals = (
    df_price.groupby(TIER_COL)
    .agg(
        total_rev=(TIER_REVENUE, 'sum'),
        total_products=(TIER_PRODUCTS, 'sum')
    )
    .reset_index()
)
tier_totals['rev_per_product'] = tier_totals['total_rev'] / tier_totals['total_products'].replace(0, np.nan)
tier_totals['rev_pct'] = (tier_totals['total_rev'] / tier_totals['total_rev'].sum() * 100).round(1)


tier_order = ['Budget', 'Low', 'Mid', 'Premium', 'Luxury']
tier_totals['sort'] = tier_totals[TIER_COL].map({t: i for i, t in enumerate(tier_order)})
tier_totals = tier_totals.sort_values('sort')

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Total Revenue by Price Tier', 'Revenue per Product by Price Tier'],
    horizontal_spacing=0.15
)

fig.add_trace(go.Bar(
    x=tier_totals[TIER_COL], y=tier_totals['total_rev'],
    marker_color='#2196F3', text=[f'${v/1e6:.0f}M' for v in tier_totals['total_rev']],
    textposition='outside', name='Total Revenue'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=tier_totals[TIER_COL], y=tier_totals['rev_per_product'],
    marker_color='#FF9800', text=[f'${v:,.0f}' for v in tier_totals['rev_per_product']],
    textposition='outside', name='Revenue/Product'
), row=1, col=2)

fig.update_layout(
    title='"Cheap Wins" — Does It? Revenue by Price Tier (All Categories)',
    template=TEMPLATE, height=500, showlegend=False
)
save_chart(fig, '01_revenue_by_tier')
fig.show()

### 3.2 — Price Tier Dominance by Category

Which tier wins in each subcategory? This is the core of the "cheap wins is false" finding —  
the winning tier is **category-dependent**.

In [7]:
dominant = (
    df_price.loc[df_price.groupby(SUBCAT_COL)[TIER_REVENUE].idxmax()]
    [[SUBCAT_COL, TIER_COL, TIER_REVENUE]]
    .copy()
)

tier_winners = dominant[TIER_COL].value_counts().reset_index()
tier_winners.columns = ['tier', 'categories_won']

fig = px.bar(
    tier_winners,
    x='tier', y='categories_won',
    title='How Many Subcategories Does Each Price Tier Dominate?',
    labels={'tier': 'Dominant Price Tier', 'categories_won': 'Number of Subcategories'},
    template=TEMPLATE,
    color='tier',
    color_discrete_map={
        'Budget': '#4CAF50', 'Mid-Range': '#2196F3',
        'Premium': '#FF9800', 'Luxury': '#E91E63'
    },
    text_auto=True
)
fig.update_layout(height=400, showlegend=False)
save_chart(fig, '02_tier_dominance_count')
fig.show()

### 3.3 — Revenue Share Stacked by Tier (Top 15 Categories)

Visualize how revenue splits across tiers within each category.

In [8]:
top15_cats = (
    df_price.groupby(SUBCAT_COL)[TIER_REVENUE].sum()
    .nlargest(15).index.tolist()
)
df_top15 = df_price[df_price[SUBCAT_COL].isin(top15_cats)].copy()

df_top15['rev_share'] = (
    df_top15.groupby(SUBCAT_COL)[TIER_REVENUE]
    .transform(lambda x: x / x.sum() * 100)
)

fig = px.bar(
    df_top15,
    x='rev_share', y=SUBCAT_COL,
    color=TIER_COL,
    orientation='h',
    title='Revenue Share by Price Tier — Top 15 Categories',
    labels={'rev_share': 'Revenue Share (%)', SUBCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_map={
    'Budget': '#4CAF50', 'Low': '#8BC34A', 'Mid': '#2196F3',
    'Premium': '#FF9800', 'Luxury': '#E91E63'
},
    barmode='stack'
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
save_chart(fig, '03_revenue_share_by_tier')
fig.show()

### 3.4 — The "Luxury Premium" Map

In which categories does luxury massively outperform budget? Where does budget actually win?  
This is the chart that kills the "cheap wins" myth.

In [9]:
df_price['rev_per_product'] = df_price[TIER_REVENUE] / df_price[TIER_PRODUCTS].replace(0, np.nan)

rpp_pivot = df_price.pivot_table(
    index=SUBCAT_COL, columns=TIER_COL,
    values='rev_per_product', aggfunc='first'
)

if 'Luxury' in rpp_pivot.columns and 'Budget' in rpp_pivot.columns:
    rpp_pivot['luxury_vs_budget'] = rpp_pivot['Luxury'] / rpp_pivot['Budget'].replace(0, np.nan)
    rpp_pivot = rpp_pivot.dropna(subset=['luxury_vs_budget'])
    rpp_pivot = rpp_pivot.sort_values('luxury_vs_budget', ascending=True)

    top_luxury = rpp_pivot.nlargest(15, 'luxury_vs_budget')
    top_budget = rpp_pivot.nsmallest(15, 'luxury_vs_budget')
    extreme = pd.concat([top_budget, top_luxury])
    extreme = extreme[extreme['luxury_vs_budget'] > 0]

    colors = ['#4CAF50' if x < 1 else '#E91E63' for x in extreme['luxury_vs_budget']]

    fig = go.Figure(go.Bar(
        x=np.log2(extreme['luxury_vs_budget']),
        y=extreme.index,
        orientation='h',
        marker_color=colors,
        text=[f'{v:.1f}×' for v in extreme['luxury_vs_budget']],
        textposition='outside'
    ))
    fig.add_vline(x=0, line_color='black', line_width=2)
    fig.update_layout(
        title='Luxury vs Budget Revenue per Product — Who Really Wins?',
        xaxis_title='Luxury ÷ Budget (log scale: 0 = equal, right = luxury wins)',
        template=TEMPLATE, height=700
    )
    save_chart(fig, '04_luxury_vs_budget')
    fig.show()
else:
    print('Tier names differ — check TIER_COL values and adapt.')
    print(df_price[TIER_COL].unique())

### 3.5 — Price vs Rating Relationship

Do more expensive products get better reviews? Or is there a backlash effect at high prices?

In [10]:
tier_rating = (
    df_price.groupby(TIER_COL)
    .agg(
        avg_rating=(TIER_AVG_RATING, 'mean'),
        total_products=(TIER_PRODUCTS, 'sum')
    )
    .reset_index()
)

fig = px.bar(
    tier_rating,
    x=TIER_COL, y='avg_rating',
    title='Average Rating by Price Tier — Does Price Buy Satisfaction?',
    labels={TIER_COL: 'Price Tier', 'avg_rating': 'Avg Rating'},
    template=TEMPLATE,
    color_discrete_sequence=['#9C27B0'],
    text_auto='.2f'
)
fig.update_yaxes(range=[3.5, 5.0])
fig.update_layout(height=400)
save_chart(fig, '05_price_vs_rating')
fig.show()

---

## 4 — Discount Effectiveness

Finding #37: Medium discounts (20–49%) lead Kitchen revenue. Finding #16: Medium beats deep across the board.  
Let's see the full picture.

In [11]:
DISC_CAT_COL = 'subcategory'         
DISC_BAND_COL = 'discount_tier'      
DISC_REVENUE = 'total_revenue'      
DISC_PRODUCTS = 'product_count'     
DISC_AVG_SALES = 'avg_units_sold'   

print(df_disc.columns.tolist())
print(f'\nDiscount bands: {df_disc[DISC_BAND_COL].unique()}')

['subcategory', 'discount_tier', 'product_count', 'avg_discount', 'avg_price', 'avg_rating', 'avg_reviews', 'total_units_sold', 'total_revenue', 'avg_revenue_per_product', 'pct_active', 'pct_best_sellers']

Discount bands: ['Light (1-19%)' 'Medium (20-49%)' 'Deep (50%+)' 'No Discount']


### 4.1 — Overall Revenue by Discount Band

Does deeper discounting drive more revenue? Or is there a sweet spot?

In [12]:
disc_totals = (
    df_disc.groupby(DISC_BAND_COL)
    .agg(
        total_rev=(DISC_REVENUE, 'sum'),
        total_products=(DISC_PRODUCTS, 'sum')
    )
    .reset_index()
)
disc_totals['rev_per_product'] = disc_totals['total_rev'] / disc_totals['total_products'].replace(0, np.nan)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Total Revenue by Discount Band', 'Revenue per Product by Discount Band'],
    horizontal_spacing=0.15
)

fig.add_trace(go.Bar(
    x=disc_totals[DISC_BAND_COL], y=disc_totals['total_rev'],
    marker_color='#4CAF50', text=[f'${v/1e6:.0f}M' for v in disc_totals['total_rev']],
    textposition='outside', name='Total Revenue'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=disc_totals[DISC_BAND_COL], y=disc_totals['rev_per_product'],
    marker_color='#FF9800', text=[f'${v:,.0f}' for v in disc_totals['rev_per_product']],
    textposition='outside', name='Revenue/Product'
), row=1, col=2)

fig.update_layout(
    title='Discount Sweet Spot — More Discount ≠ More Revenue',
    template=TEMPLATE, height=500, showlegend=False
)
save_chart(fig, '06_revenue_by_discount')
fig.show()

### 4.2 — Discount Effectiveness Curve

Revenue per product as a function of discount depth. Where's the peak?

In [13]:
fig = px.line(
    disc_totals.sort_values(DISC_BAND_COL),
    x=DISC_BAND_COL, y='rev_per_product',
    title='The Discount Curve — Revenue per Product vs Discount Depth',
    labels={DISC_BAND_COL: 'Discount Band', 'rev_per_product': 'Revenue per Product ($)'},
    template=TEMPLATE,
    markers=True
)
fig.update_traces(line=dict(width=3, color='#E91E63'))
fig.update_layout(height=400)
save_chart(fig, '07_discount_curve')
fig.show()

### 4.3 — Discount Effectiveness by Category

The discount sweet spot varies by category — just like pricing. Let's see which categories benefit most from discounting.

In [14]:
disc_dominant = (
    df_disc.loc[df_disc.groupby(DISC_CAT_COL)[DISC_REVENUE].idxmax()]
    [[DISC_CAT_COL, DISC_BAND_COL, DISC_REVENUE]]
    .copy()
)

band_winners = disc_dominant[DISC_BAND_COL].value_counts().reset_index()
band_winners.columns = ['band', 'categories_won']

fig = px.bar(
    band_winners,
    x='band', y='categories_won',
    title='Which Discount Band Dominates the Most Categories?',
    labels={'band': 'Winning Discount Band', 'categories_won': 'Number of Subcategories'},
    template=TEMPLATE,
    color_discrete_sequence=['#00BCD4'],
    text_auto=True
)
fig.update_layout(height=400)
save_chart(fig, '08_discount_winners')
fig.show()

### 4.4 — Discount Revenue Share (Top 15 Categories)

Stacked bars: how does revenue split across discount bands within each top category?

In [15]:
top15_disc = (
    df_disc.groupby(DISC_CAT_COL)[DISC_REVENUE].sum()
    .nlargest(15).index.tolist()
)
df_disc_top = df_disc[df_disc[DISC_CAT_COL].isin(top15_disc)].copy()

df_disc_top['rev_share'] = (
    df_disc_top.groupby(DISC_CAT_COL)[DISC_REVENUE]
    .transform(lambda x: x / x.sum() * 100)
)

fig = px.bar(
    df_disc_top,
    x='rev_share', y=DISC_CAT_COL,
    color=DISC_BAND_COL,
    orientation='h',
    title='Discount Revenue Share — Top 15 Categories',
    labels={'rev_share': 'Revenue Share (%)', DISC_CAT_COL: ''},
    template=TEMPLATE,
    barmode='stack'
)
fig.update_layout(height=600, yaxis={'categoryorder': 'total ascending'})
save_chart(fig, '09_discount_share_by_category')
fig.show()

### 4.5 — No Discount vs Discounted

The most basic question: do discounted products outperform full-price ones?

In [16]:
df_disc['is_discounted'] = ~df_disc[DISC_BAND_COL].str.contains('0|No|None|Full', case=False, na=False)

disc_vs_full = (
    df_disc.groupby('is_discounted')
    .agg(
        total_rev=(DISC_REVENUE, 'sum'),
        total_products=(DISC_PRODUCTS, 'sum')
    )
    .reset_index()
)
disc_vs_full['rev_per_product'] = disc_vs_full['total_rev'] / disc_vs_full['total_products'].replace(0, np.nan)
disc_vs_full['label'] = disc_vs_full['is_discounted'].map({True: 'Discounted', False: 'Full Price'})

fig = px.bar(
    disc_vs_full,
    x='label', y='rev_per_product',
    title='Revenue per Product: Discounted vs Full Price',
    labels={'label': '', 'rev_per_product': 'Revenue per Product ($)'},
    template=TEMPLATE,
    color='label',
    color_discrete_map={'Discounted': '#4CAF50', 'Full Price': '#FF9800'},
    text_auto=',.0f'
)
fig.update_layout(height=400, showlegend=False)
save_chart(fig, '10_discounted_vs_full')
fig.show()

---

## 5 — Price Sweet Spot Deep Dive

For specific high-value categories, show the price distribution curve alongside where revenue concentrates.

In [17]:
focus_cats = df_price.groupby(SUBCAT_COL)[TIER_REVENUE].sum().nlargest(4).index.tolist()
print(f'Focus categories: {focus_cats}')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=focus_cats,
    horizontal_spacing=0.12,
    vertical_spacing=0.15
)

for i, cat in enumerate(focus_cats):
    row, col = i // 2 + 1, i % 2 + 1
    cat_data = df_price[df_price[SUBCAT_COL] == cat].copy()
    cat_data['rev_per_product'] = cat_data[TIER_REVENUE] / cat_data[TIER_PRODUCTS].replace(0, np.nan)
    
    fig.add_trace(go.Bar(
        x=cat_data[TIER_COL],
        y=cat_data['rev_per_product'],
        marker_color=['#4CAF50', '#2196F3', '#FF9800', '#E91E63'][:len(cat_data)],
        showlegend=False,
        text=[f'${v:,.0f}' for v in cat_data['rev_per_product']],
        textposition='outside'
    ), row=row, col=col)

fig.update_layout(
    title='Price Sweet Spot by Category — Revenue per Product',
    template=TEMPLATE, height=700
)
save_chart(fig, '11_price_sweet_spot_detail')
fig.show()

Focus categories: ['Kitchen & Dining', 'Hair Care Products', 'Home Storage & Organization', 'Toys & Games']


---

## 6 — Key Findings

In [18]:
print('=' * 60)
print('PRICING & DISCOUNT ANALYSIS — KEY FINDINGS')
print('=' * 60)

best_tier = tier_totals.loc[tier_totals['total_rev'].idxmax()]
print(f'\n1. HIGHEST REVENUE TIER: {best_tier[TIER_COL]}')
print(f'   Total: ${best_tier["total_rev"]/1e6:.0f}M ({best_tier["rev_pct"]}% of all revenue)')

best_rpp = tier_totals.loc[tier_totals['rev_per_product'].idxmax()]
print(f'\n2. HIGHEST REVENUE/PRODUCT: {best_rpp[TIER_COL]}')
print(f'   ${best_rpp["rev_per_product"]:,.0f} per product')

print(f'\n3. TIER DOMINANCE ACROSS CATEGORIES:')
for _, row in tier_winners.iterrows():
    print(f'   {row["tier"]}: wins {row["categories_won"]} subcategories')

best_disc = disc_totals.loc[disc_totals['rev_per_product'].idxmax()]
print(f'\n4. BEST DISCOUNT BAND: {best_disc[DISC_BAND_COL]}')
print(f'   ${best_disc["rev_per_product"]:,.0f} revenue per product')

print(f'\n5. DISCOUNT BAND DOMINANCE:')
for _, row in band_winners.iterrows():
    print(f'   {row["band"]}: wins {row["categories_won"]} subcategories')

print('\n' + '=' * 60)

PRICING & DISCOUNT ANALYSIS — KEY FINDINGS

1. HIGHEST REVENUE TIER: Low
   Total: $1513M (32.5% of all revenue)

2. HIGHEST REVENUE/PRODUCT: Luxury
   $6,624 per product

3. TIER DOMINANCE ACROSS CATEGORIES:
   Low: wins 111 subcategories
   Mid: wins 59 subcategories
   Luxury: wins 55 subcategories
   Premium: wins 17 subcategories
   Budget: wins 6 subcategories

4. BEST DISCOUNT BAND: No Discount
   $7,903 revenue per product

5. DISCOUNT BAND DOMINANCE:
   Light (1-19%): wins 144 subcategories
   Medium (20-49%): wins 98 subcategories
   Deep (50%+): wins 4 subcategories
   No Discount: wins 1 subcategories



In [19]:
con.close()
print('Done. DuckDB connection closed.')
print(f'Charts saved to: {CHARTS_DIR}/')

Done. DuckDB connection closed.
Charts saved to: charts/04_pricing_discounts/
